# Create MERlin scripts

Generates the per-experiment MERlin config/run files into `SAMPLE_DIR/merlin/` — replacing the old shared cluster location (`~/Software/merfish-parameters/`). Codebook and microscope-parameters files are NOT copied here: they're shared reference data shipped in `MERci/data/configs/merlin/{codebooks,microscope}/`, and since this `MERci/` clone already lives inside `SAMPLE_DIR/`, the slurm script below references them by their path inside this clone directly — self-contained, no separate cluster-side copy step.

Run this after notebook 06 (needs `experiment_info.yaml`).

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

MERCI_DIR    = Path(os.getcwd()).parent.parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/prepare_imaging/<variant>/<acquisition>/)
SAMPLE_DIR   = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_26/
METADATA_DIR = SAMPLE_DIR / "metadata"
POSITIONS_DIR = SAMPLE_DIR / "positions"
MERLIN_DIR   = SAMPLE_DIR / "merlin"             # new per-experiment MERlin folder
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.experiment_info import load_experiment_info
from MERci.acquisition.merlin_config import (
    resolve_codebook_filename, resolve_microscope_parameters_filename,
    MerlinAnalysisSpec, create_merlin_analysis_parameters,
    create_cluster_resource_allocation, create_snakemake_parameters,
    resolve_cluster_sample_dir, create_slurm_submit_script,
)

info = load_experiment_info(METADATA_DIR / "experiment_info.yaml")
MICROSCOPE  = info.microscope

# LOCAL_NAME is this acquisition's own local file-naming tag (SAMPLE_DIR.name)
# -- matches what notebooks 01-05 already used to name positions_*.txt /
# data_organization_*.csv / etc., so this notebook's file lookups (and its own
# generated merlin/ filenames) stay consistent with what's already on disk.
# SAMPLE_NAME is the TRUE top-level experiment id (from experiment_info.yaml,
# notebook 06) -- needed only for resolve_cluster_sample_dir's project-root/
# cluster-path resolution below.
LOCAL_NAME  = SAMPLE_DIR.name
SAMPLE_NAME = info.sample_name

print(f"LOCAL_NAME  : {LOCAL_NAME}")
print(f"SAMPLE_NAME : {SAMPLE_NAME}")
print(f"MICROSCOPE  : {MICROSCOPE}")
print(f"lib_name    : {info.lib_name}")
print(f"data_home   : {info.data_home}")
print(f"merlin_home : {info.merlin_home}")
print(f"folder_name : {info.folder_name}")

## Resolve shared reference files (codebook, microscope parameters)

These live in `MERci/data/configs/merlin/` — shipped with the repo, not regenerated per experiment. As a sanity check, the codebook's own `bit_names` row is compared against this experiment's bit count from `round_bit_color_map.csv`, to catch a mismatched codebook before it reaches the cluster.

In [ ]:
codebook_path   = MERCI_DIR / "data" / "configs" / "merlin" / "codebooks" / resolve_codebook_filename(info.lib_name)
microscope_path = MERCI_DIR / "data" / "configs" / "merlin" / "microscope" / resolve_microscope_parameters_filename(MICROSCOPE)

for p in (codebook_path, microscope_path):
    if not p.exists():
        raise FileNotFoundError(
            f"{p} not found -- add it to MERci/data/configs/merlin/ before running this notebook."
        )

rbc_path = METADATA_DIR / "round_bit_color_map.csv"
if rbc_path.exists():
    # Distinct bit count (NOT round.max() -- a round can carry several bits,
    # one per color, so round.max() is the hyb-round count, not the bit count).
    n_bits = int(pd.read_csv(rbc_path)["bit"].nunique())
    codebook_bit_names_line = next(
        line for line in codebook_path.read_text().splitlines() if line.startswith("bit_names")
    )
    codebook_n_bits = len(codebook_bit_names_line.split(",")) - 1
    if codebook_n_bits != n_bits:
        print(f"WARNING: codebook {codebook_path.name} has {codebook_n_bits} bits, "
              f"but this experiment images {n_bits} -- double-check lib_name/codebook choice.")
    else:
        print(f"Bit count OK: {n_bits} bits, matching {codebook_path.name}.")
else:
    n_bits = None
    print(f"WARNING: {rbc_path} not found -- run notebook 03 first.")

print(f"Codebook  : {codebook_path}")
print(f"Microscope: {microscope_path}")

## MERlin analysis-parameters JSON

Built from a compact `MerlinAnalysisSpec` (which steps to include) instead of copying and hand-editing a prior experiment's file. Adjust the spec below to match this experiment; see `MerlinAnalysisSpec`'s docstring for every tunable field.

In [ ]:
spec = MerlinAnalysisSpec(
    n_optimize_iterations = info.extra.get("n_opt", 10),   # MERlin param, independent of bit count
    include_reporting     = True,
    include_segmentation  = False,   # True to append a cell-segmentation chain
    segmentation_method   = "CellPoseSegment3D",   # or "CellPoseSegmentSAM"
)

analysis_path = MERLIN_DIR / "analysis" / f"merlin_analysis_{LOCAL_NAME}.json"
create_merlin_analysis_parameters(spec, analysis_path)
print(f"Saved: {analysis_path}")

## Snakemake cluster-resource-allocation + parameters JSON

In [ ]:
cluster_template = MERCI_DIR / "data" / "configs" / "merlin" / "snakemake" / "cluster_resource_allocation_basic.json"

cluster_resource_path = MERLIN_DIR / "snakemake" / f"cluster_resource_allocation_{LOCAL_NAME}.json"
create_cluster_resource_allocation(
    template_path=cluster_template, exp_name=LOCAL_NAME,
    n_optimize_iterations=spec.n_optimize_iterations, output_path=cluster_resource_path,
)
print(f"Saved: {cluster_resource_path}")

snakemake_params_path = MERLIN_DIR / "snakemake" / f"parameters_{LOCAL_NAME}.json"
create_snakemake_parameters(
    exp_name=LOCAL_NAME, cluster_config_path=cluster_resource_path,
    output_path=snakemake_params_path,
)
print(f"Saved: {snakemake_params_path}")

## Slurm submit script

Every path in the generated script is written relative to one `$SAMPLE_DIR` bash variable (this experiment's acquisition root — see `resolve_cluster_sample_dir`), instead of five separately-resolved absolute paths. This makes the script identical whether it was generated here (Windows, before transferring `SAMPLE_DIR` to the cluster — `$SAMPLE_DIR` is then *predicted* from the sample name) or regenerated on the cluster itself after the transfer (Linux — `$SAMPLE_DIR` is just the real current path, no guessing).

In [ ]:
data_org_path  = METADATA_DIR / f"data_organization_{MICROSCOPE.upper()}_{LOCAL_NAME}.csv"
positions_path = POSITIONS_DIR / f"positions_{LOCAL_NAME}.txt"
for p in (data_org_path, positions_path):
    if not p.exists():
        print(f"WARNING: {p} not found. For a multi-boundary experiment, set the "
              f"correct per-segment file path manually before submitting to the cluster.")

# $SAMPLE_DIR in the generated script: this experiment's acquisition root as it
# will be addressed from the Linux cluster. Predicted from the TRUE sample_name +
# imaging_dir (experiment_info.yaml, notebook 06) when running on Windows
# (before transfer); the real current path when already running on the
# cluster (Linux) -- see resolve_cluster_sample_dir.
sample_dir_cluster = resolve_cluster_sample_dir(SAMPLE_DIR, SAMPLE_NAME, info.extra.get("imaging_dir", ""))

def _rel(p):
    """POSIX path relative to SAMPLE_DIR, for use as "$SAMPLE_DIR/<...>" in the script."""
    return Path(p).relative_to(SAMPLE_DIR).as_posix()

submit_path = MERLIN_DIR / "slurm" / "submit" / f"merlin_slurm_{LOCAL_NAME}.sh"
create_slurm_submit_script(
    label                   = LOCAL_NAME,
    sample_dir              = sample_dir_cluster,
    parameters_file         = _rel(snakemake_params_path),
    analysis_file           = _rel(analysis_path),
    data_organization_file  = _rel(data_org_path),
    positions_file          = _rel(positions_path),
    codebook_file           = _rel(codebook_path),
    microscope_file         = _rel(microscope_path),
    data_home               = info.data_home,
    folder_name             = info.folder_name,
    output_path             = submit_path,
)
print(f"$SAMPLE_DIR (cluster) : {sample_dir_cluster}")
print(f"Saved: {submit_path}\n")
print(submit_path.read_text())